# Deep Agents (LangChain) + LangSmith Tracing — Hands-On Tutorial

Companion to the CS357 activity *Deep Agents: Planning, Subagents, and a Virtual Filesystem* and the graded **Deep Agent lab**.

A **deep agent** is LangChain's batteries-included harness for *long, multi-step* tasks. On top of a normal tool-calling loop it adds four things a marathon task needs:

1. **Planning** — a `write_todos` tool the agent uses to keep a revisable plan.
2. **Virtual filesystem** — `ls / read_file / write_file / edit_file / glob / grep` used as working memory *outside* the context window.
3. **Subagents** — a `task` tool that spawns an ephemeral agent with a fresh context that returns one report.
4. **Long-term memory** — a store that can persist across runs.

We build one with `create_deep_agent`, then turn on **LangSmith** so we can *see* the plan, the file writes, and the subagent calls as a trace.

Steps: (0) setup → (1) a baseline flat agent → (2) a deep agent → (3) inspect the filesystem & plan → (4) LangSmith tracing → (5) exercises.

Runs against a **local Ollama** model (no paid API required); LangSmith tracing is optional but recommended.

## 0. Environment & Dependencies

- `deepagents` — the harness (`create_deep_agent`).
- `langchain`, `langgraph` — deep agents compile to a LangGraph graph.
- `langchain-ollama` — local model integration.
- `langsmith` — tracing/observability.

Have an **Ollama server** running with a tool-capable model pulled (e.g. `ollama pull llama3.1`).

In [ ]:
!pip install -q deepagents langchain langgraph langchain-ollama langsmith

In [ ]:
import os

# Point at your Ollama server (local or remote). Override OLLAMA_BASE_URL if needed.
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1")

# --- LangSmith tracing (optional but recommended) ---
# Get a key at https://smith.langchain.com and set it before running.
# If LANGSMITH_API_KEY is unset, the notebook still runs; you just get no trace.
if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "cs357-deepagents")
    print("LangSmith tracing ON -> project", os.environ["LANGSMITH_PROJECT"])
else:
    print("LangSmith tracing OFF (set LANGSMITH_API_KEY to enable).")

print("Ollama:", OLLAMA_BASE_URL, "model:", OLLAMA_MODEL)

## 1. A baseline FLAT agent (the marathon problem)

First, a plain tool-calling agent with one tool. Give it a long, multi-part task and notice how everything it does must fit in a single context window — the exact limitation deep agents address.

In [ ]:
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)

def word_count(text: str) -> str:
    """Return the number of words in the given text."""
    return str(len(text.split()))

flat_agent = create_react_agent(llm, tools=[word_count])

task = (
    "Outline a 3-section report on local vs hosted LLMs, then for EACH section "
    "write one paragraph, then count the words in section 2."
)
flat_result = flat_agent.invoke({"messages": [{"role": "user", "content": task}]})
print(flat_result["messages"][-1].content)

## 2. The same task as a DEEP agent

`create_deep_agent(model, tools, instructions)` gives you the four pillars for free. Notice that **planning, files, and subagents are invoked by the *instructions*** — your prose tells the agent to use the built-in tools.

In [ ]:
from deepagents import create_deep_agent

def web_search(query: str) -> str:
    """Stub search tool. Replace with a real search API for non-trivial work."""
    return f"(pretend results for: {query})"

agent = create_deep_agent(
    model=llm,                      # any LangChain-compatible model; local Ollama here
    tools=[web_search, word_count], # YOUR domain tools...
    instructions=(                  # ...the built-ins (plan, files, subagents) come free
        "You are a research assistant. FIRST call write_todos to make a plan. "
        "Save notes and each section to files with write_file. "
        "Use a subagent (the task tool) to summarize any long source. "
        "Finally write report.md and revise it against your plan."
    ),
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Research local vs hosted LLM tradeoffs and write report.md (3 sections)."}]})
print(result["messages"][-1].content)

## 3. Inspect the plan and the virtual filesystem

Deep-agent file tools write to a virtual filesystem carried in the run **state**, not your real disk. After a run, the files the agent created appear under the `files` key of the returned state. This is the *context offloading* in action — bulky work lived in files, not the context window.

In [ ]:
# The virtual filesystem is returned in the agent state under 'files'.
files = result.get("files", {})
print("Files the agent created:", list(files.keys()))
if "report.md" in files:
    print("\n--- report.md ---\n")
    print(files["report.md"])

## 4. Read the trace in LangSmith

If you set `LANGSMITH_API_KEY` in step 0, every model call, tool call, and subagent from the run above was recorded. Open <https://smith.langchain.com>, select the `cs357-deepagents` project, and open the latest run.

A healthy trace for this task shows, as a nested tree:

- a `write_todos` call near the top (the **plan**),
- a series of tool calls — `web_search`, `write_file` (the **filesystem**),
- one or more `task` spans, each with their **own** nested calls (the **subagents**),
- a final write/revise of `report.md`.

**Debugging practice:** if `report.md` is weak, do not re-read only the output — read the trace. Did the plan match the task? Did a subagent return a thin report? Where did the tokens go? (This is the *Observability* activity's three pillars — logs, metrics, traces — applied to a deep agent; LangSmith is the LangChain-native sibling of the OpenTelemetry/Jaeger stack from the observability lab.)

> **Privacy:** a trace can capture raw prompts and tool inputs. Do not send secrets or real personal data through a traced agent without considering retention.

## 5. Exercises

1. **Plan inspection.** Add `print` statements (or read the trace) to capture the agent's `write_todos` plan. Did its plan match the task? Edit the `instructions` to force a 5-step plan and compare.
2. **Subagent quality.** Change the instructions so the subagent is told to return *at least 5 bullet points*. Re-run and confirm in the trace that the subagent's report got richer.
3. **Files vs subagent.** Give the agent a task where writing to a file is clearly better than spawning a subagent, and one where the reverse holds. Justify each in a markdown cell.
4. **Flat vs deep.** Run the *same* long task through the flat agent (step 1) and the deep agent (step 2). Compare completeness and explain the difference in terms of the context window.
5. **When NOT to.** Run a trivial one-shot question through the deep agent and measure the extra latency/tokens. Write one sentence on when a deep agent is the wrong tool.